In [10]:
# Standard library imports
import os
import sys
import math
import random
from collections import Counter, defaultdict
from typing import List
from random import sample as random_sample

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tqdm import tqdm

# PyTorch imports
import torch
import torch.nn as nn
from torch import Tensor, no_grad, argmax
from torch import max as torch_max
from torch.nn import CrossEntropyLoss
from torch.optim import SGD, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, CosineAnnealingWarmRestarts
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from torchsummary import summary
import torchvision.transforms as transforms
from torchvision.transforms import functional as F

# Ignite imports
from ignite.engine import Events, create_supervised_trainer, create_supervised_evaluator
from ignite.handlers import ReduceLROnPlateauScheduler
from ignite.metrics import Accuracy, Fbeta, Loss, Precision, Recall


# files import
import src.dataloader as dataloader
from src.dataloader import PeopleDataset

from src.utils.engine import setup_trainer, setup_evaluators, train_epoch_and_get_metrics_dict, calculate_epoch_metrics
from src.utils.logging import setup_metrics_history, add_metrics_to_history, print_epoch_summary, save_best_models
from src.utils import plotting

In [13]:


def train_model(config, model_class, class_exclusion_threshold=None, classes_to_exclude=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")

    """Preparing the data"""
    train_transforms = dataloader.get_transforms(augmentation_type=config.TRAIN_AUGMENTATION_TYPE)
    valid_transforms = dataloader.get_transforms(augmentation_type=config.VALID_AUGMENTATION_TYPE)

    print("Loading the dataset...")
    full_dataset = PeopleDataset(config.PATH_TO_DATA)
    full_dataset.print_class_distribution()

    if class_exclusion_threshold:
        print("Removing rare classes")
        # Option 1: Filter by minimum threshold of class in dataset
        full_dataset.filter_by_min_threshold(min_threshold=class_exclusion_threshold)

    if classes_to_exclude:
        print("Excluding selected classes")
        # Option 2: Filter by explicitly excluding class names
        full_dataset.filter_by_classes(classes_to_exclude=classes_to_exclude)

    if classes_to_exclude or class_exclusion_threshold:
        # Rebuild class_to_index AFTER filtering
        full_dataset.class_names = sorted(
            list(set(full_dataset.labels)))  # Get unique remaining labels (which are strings) and sort them
        full_dataset.class_to_index = {cls_name: i for i, cls_name in enumerate(full_dataset.class_names)}
        print(f"Number of classes after filtering: {len(full_dataset.class_names)}")  # Verify the number of classes

    train_set, valid_set = dataloader.split_dataset(full_dataset, valid_ratio=0.2)
    train_set.dataset.transform = train_transforms
    valid_set.dataset.transform = valid_transforms

    # Showing first 12 images after transforming them
    # plotting.show_first_images(full_dataset)

    print(f"Setting up data loaders with batch_size={config.BATCH_SIZE}...")
    train_loader, valid_loader = dataloader.setup_data_loaders(
        batch_size=config.BATCH_SIZE,
        train_set=train_set,
        valid_set=valid_set
    )

    """Training setup"""
    num_classes = len(full_dataset.class_names)  # Use the updated class_names
    print(f"Number of classes: {num_classes}")

    model = model_class(num_classes=num_classes)
    model.to(device)
    summary(model, (3, 288, 512))
    print("\n")

    train_indices = train_set.indices
    train_targets = [full_dataset[idx][1] for idx in train_indices]
    class_counts = [train_targets.count(i) for i in range(num_classes)]
    class_counts = [c if c != 0 else 1 for c in class_counts]
    class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_weights / class_weights.sum()
    class_weights = class_weights.to(device)

    optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

    if config.SCHEDULER == 'CosineAnnealingWarmRestarts':
        scheduler = CosineAnnealingWarmRestarts(optimizer=optimizer, T_0=25, eta_min=1e-6)
    elif config.SCHEDULER == 'CosineAnnealingLR':
        scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=config.NUM_EPOCHS, eta_min=0)

    criterion = CrossEntropyLoss(weight=class_weights.to(device))

    print("Initializing trainer and evaluators...")
    trainer = setup_trainer(model, optimizer, criterion, device)
    train_evaluator, valid_evaluator = setup_evaluators(model, criterion, device)

    train_metrics_history, valid_metrics_history = setup_metrics_history()

    best_valid_loss = float('inf')
    best_valid_f1 = 0.0
    model_name = model.__class__.__name__

    print(f"\nStarting training for {config.NUM_EPOCHS} epochs...")

    for epoch in range(config.NUM_EPOCHS):
        print(f"\nEpoch {epoch + 1}/{config.NUM_EPOCHS}")

        train_metrics_dict = train_epoch_and_get_metrics_dict(model, train_loader, criterion, optimizer, device, epoch,
                                                              config.NUM_EPOCHS)
        scheduler.step()
        add_metrics_to_history(train_metrics_history, train_metrics_dict)

        valid_metrics_dict = {}
        if valid_loader:
            valid_metrics_dict = calculate_epoch_metrics(model, valid_loader, criterion, device)
            add_metrics_to_history(valid_metrics_history, valid_metrics_dict)
            best_valid_loss, best_valid_f1 = save_best_models(
                current_metrics=valid_metrics_dict,
                model=model,
                model_name=model_name,
                best_loss=best_valid_loss,
                best_f1=best_valid_f1,
                save_dir=config.SAVE_DIR
            )

        print_epoch_summary(epoch, train_metrics_dict, valid_metrics_dict)

    """Results visualization"""
    print("\nTraining completed!")
    print(f"Results location: {config.RESULT_DIR}")
    print("Saving results...")
    metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'loss']
    plotting.plot_metrics(train_metrics_history, valid_metrics_history, metrics_to_plot, save_path=config.RESULT_DIR)

    # To plot loss and one metric
    # plot_metric_and_loss(train_metrics_history, valid_metrics_history, "accuracy")

    class_names = full_dataset.class_names  # Use the updated class_names
    # plotting.visualize_predictions(model, valid_loader, device, class_names)

    print(f"\nPlotting metrics per class...")
    plotting.plot_metrics_per_class(model, valid_loader, device, class_names, save_path=config.RESULT_DIR)

    # evaluate_model(model, test_loader, criterion, device)

In [ ]:
# simulating config file as class

class Config:
    def __init__(self):
        self.PATH_TO_DATA = "PATH_TO_YOUR_DATA"
        self.RESULT_DIR = "RESULTS_SAVING_DIRECTORY"
        self.SAVE_DIR = "WEIGHTS_SAING_DIRECTORY"

        self.BATCH_SIZE = 32
        self.LEARNING_RATE = 0.001
        self.MOMENTUM = 0.9
        self.NUM_EPOCHS = 75

        self.SCHEDULER = 'CosineAnnealingLR'

        self.WEIGHT_DECAY = 3e-3

        self.TRAIN_AUGMENTATION_TYPE = "basic"
        self.VALID_AUGMENTATION_TYPE = None

config = Config()

In [16]:
from src.models.__all_models import *

model_to_train = PoseCNNsc_13_24_35

# Call the training function from trainer.py
train_model(config, model_to_train)

Using device: cuda

Augmentation type: basic
Augmentation type: None
Loading the dataset...
Class Distribution:
  Class 'sports': 2512 samples (0.2031)
  Class 'inactivity quiet/light': 135 samples (0.0109)
  Class 'miscellaneous': 530 samples (0.0429)
  Class 'occupation': 1680 samples (0.1358)
  Class 'water activities': 752 samples (0.0608)
  Class 'home activities': 982 samples (0.0794)
  Class 'lawn and garden': 800 samples (0.0647)
  Class 'religious activities': 0 samples (0.0000)
  Class 'winter activities': 533 samples (0.0431)
  Class 'conditioning exercise': 1223 samples (0.0989)
  Class 'bicycling': 390 samples (0.0315)
  Class 'fishing and hunting': 528 samples (0.0427)
  Class 'dancing': 469 samples (0.0379)
  Class 'walking': 442 samples (0.0357)
  Class 'running': 228 samples (0.0184)
  Class 'self care': 0 samples (0.0000)
  Class 'home repair': 753 samples (0.0609)
  Class 'volunteer activities': 0 samples (0.0000)
  Class 'music playing': 410 samples (0.0332)
  Class

Validation: 100%|██████████| 78/78 [00:32<00:00,  2.37batch/s, val_loss=0.0278]


Saved new best model(s) - Loss: 2.4469 | F1: 0.1491

Epoch 1 Summary:
Train - Loss: 2.6616, Acc: 13.03%, Precision: 0.1714, Recall: 0.1303, F1: 0.1205
Valid - Loss: 2.4469, Acc: 18.36%, Precision: 0.2208, Recall: 0.1836, F1: 0.1491

Epoch 2/75


Validation: 100%|██████████| 78/78 [00:38<00:00,  2.02batch/s, val_loss=0.0297]


Saved new best model(s) - Loss: 2.3670 | F1: 0.1663

Epoch 2 Summary:
Train - Loss: 2.5186, Acc: 15.74%, Precision: 0.2162, Recall: 0.1574, F1: 0.1468
Valid - Loss: 2.3670, Acc: 21.43%, Precision: 0.2455, Recall: 0.2143, F1: 0.1663

Epoch 3/75


...

Saved new best model(s) - F1: 0.5359

Epoch 73 Summary:
Train - Loss: 1.1169, Acc: 52.85%, Precision: 0.5754, Recall: 0.5285, F1: 0.5133
Valid - Loss: 1.3980, Acc: 54.55%, Precision: 0.5867, Recall: 0.5455, F1: 0.5359

Epoch 74/75


Validation: 100%|██████████| 78/78 [00:26<00:00,  2.91batch/s, val_loss=0.0131] 



Epoch 74 Summary:
Train - Loss: 1.1212, Acc: 52.56%, Precision: 0.5714, Recall: 0.5256, F1: 0.5088
Valid - Loss: 1.3821, Acc: 53.62%, Precision: 0.5835, Recall: 0.5362, F1: 0.5238

Epoch 75/75


Validation: 100%|██████████| 78/78 [00:27<00:00,  2.83batch/s, val_loss=0.0127] 



Epoch 75 Summary:
Train - Loss: 1.1271, Acc: 52.61%, Precision: 0.5737, Recall: 0.5261, F1: 0.5126
Valid - Loss: 1.3832, Acc: 53.86%, Precision: 0.5805, Recall: 0.5386, F1: 0.5278

Training completed!
Results location: V:\ML\yandex-ml-2025\src\saves\PoseCNNsc_13_24_35_#result1
Saving results...
saved metrics

Plotting metrics per class...
saved distribustion
